# Eval — baseline LoRA on clean holdout (no reference adapter)

Minimal: setup deps → metric utility (custom vLLM/torch) → load Nemotron + our LoRA → generate on holdout → save predictions.csv. Skips the reference-adapter comparison from the official demo because Kaggle doesn't mount user-kernel outputs as kernel_sources reliably.

In [ ]:
ADAPTER_PATH = '/kaggle/input/notebooks/gastonz195/train-structured-v1/submission_adapter'
import os
for cand in [ADAPTER_PATH,
             '/kaggle/input/notebooks/gastonz195/train-structured-v1/sft_adapter',
             '/kaggle/usr/lib/notebooks/gastonz195/train_structured_v1/submission_adapter']:
    if os.path.isdir(cand) and os.path.exists(os.path.join(cand, 'adapter_config.json')):
        ADAPTER_PATH = cand
        break
else:
    raise FileNotFoundError(f'no adapter found; check /kaggle/input/notebooks/gastonz195/')
print('ADAPTER_PATH:', ADAPTER_PATH)
print('files:', os.listdir(ADAPTER_PATH))

In [ ]:
import sys, glob, subprocess, site, shutil, stat

triton_wheels = sorted(glob.glob('/kaggle/input/**/triton-3.6.0*.whl', recursive=True))
assert triton_wheels, 'no triton 3.6.0 wheel found'
target = '/kaggle/working/pydeps'
os.makedirs(target, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--target', target,
                '--upgrade', '--ignore-installed', triton_wheels[0]], check=True)
sys.path.insert(0, target); site.addsitedir(target)
print('triton installed')

ptxas_src_root = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script'
if os.path.isdir(ptxas_src_root):
    subprocess.run('tar -cf - -C ' + ptxas_src_root + ' . | tar -xf - -C /tmp', shell=True, check=True)
    for p in ['/tmp/triton/backends/nvidia/bin/ptxas',
              '/tmp/triton/backends/nvidia/bin/ptxas-blackwell']:
        if os.path.exists(p):
            os.chmod(p, os.stat(p).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    os.environ['TRITON_PTXAS_PATH'] = '/tmp/triton/backends/nvidia/bin/ptxas'
    print('ptxas configured')
else:
    print('ryanholbrook utility not available; relying on default Pro 6000 ptxas')

causal_wheel = sorted(glob.glob('/kaggle/input/**/causal_conv1d*.whl', recursive=True))[-1]
mamba_wheel = sorted(glob.glob('/kaggle/input/**/mamba_ssm*.whl', recursive=True))[-1]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', causal_wheel], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', mamba_wheel], check=True)
print('mamba_ssm + causal-conv1d installed')

In [ ]:
metric_root = '/kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script'
assert os.path.isdir(metric_root), 'metric utility not found'
subprocess.run('uv pip uninstall torch torchvision torchaudio', shell=True, check=False)
subprocess.run(f'tar -cf - -C {metric_root} . | tar -xf - -C /tmp', shell=True, check=True)
for p in ['/tmp/triton/backends/nvidia/bin/ptxas', '/tmp/triton/backends/nvidia/bin/ptxas-blackwell']:
    if os.path.exists(p):
        os.chmod(p, os.stat(p).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
os.environ['TRITON_PTXAS_PATH'] = '/tmp/triton/backends/nvidia/bin/ptxas'
sys.path.insert(0, '/tmp')
print('metric utility extracted; /tmp prepended to sys.path')

In [ ]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import kagglehub
MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
print('MODEL_PATH:', MODEL_PATH)

from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

llm = LLM(
    model=str(MODEL_PATH),
    tensor_parallel_size=1,
    max_num_seqs=64,
    gpu_memory_utilization=0.85,
    dtype='auto',
    max_model_len=8192,
    trust_remote_code=True,
    enable_lora=True,
    max_lora_rank=32,
    enable_prefix_caching=True,
    enable_chunked_prefill=True,
)
sampling = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=7680)
tokenizer = llm.get_tokenizer()
print('vLLM ready')

In [ ]:
import pandas as pd
HOLDOUT_CANDIDATES = [
    '/kaggle/input/datasets/gastonz195/nemotron-holdout-clean-v1/holdout_clean.csv',
    '/kaggle/input/nemotron-holdout-clean-v1/holdout_clean.csv',
]
HOLDOUT_PATH = next((p for p in HOLDOUT_CANDIDATES if os.path.exists(p)), None)
assert HOLDOUT_PATH, 'holdout not found'
df = pd.read_csv(HOLDOUT_PATH)
print(f'holdout: {len(df)} rows')

SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
prompts = []
for prompt_text in df['prompt']:
    try:
        p = tokenizer.apply_chat_template(
            [{'role': 'user', 'content': prompt_text + SUFFIX}],
            tokenize=False, add_generation_prompt=True, enable_thinking=True,
        )
    except TypeError:
        p = tokenizer.apply_chat_template(
            [{'role': 'user', 'content': prompt_text + SUFFIX}],
            tokenize=False, add_generation_prompt=True,
        )
    prompts.append(p)

In [ ]:
import time
t0 = time.time()
outputs = llm.generate(
    prompts,
    sampling_params=sampling,
    lora_request=LoRARequest('adapter', 1, ADAPTER_PATH),
)
print(f'generated {len(outputs)} in {(time.time()-t0)/60:.1f} min')

In [ ]:
import re
def extract_boxed(text):
    if text is None: return 'NOT_FOUND'
    m = re.findall(r'\\boxed\{([^}]*)(?:\}|$)', text)
    if m:
        ne = [x.strip() for x in m if x.strip()]
        return ne[-1] if ne else m[-1].strip()
    for pat in [r'The final answer is:\s*([^\n]+)', r'Final answer\s*[:：]\s*([^\n]+)']:
        mm = re.findall(pat, text, re.IGNORECASE)
        if mm: return mm[-1].strip()
    nums = re.findall(r'-?\d+(?:\.\d+)?', text)
    if nums: return nums[-1]
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return lines[-1] if lines else 'NOT_FOUND'

df['raw_output'] = [o.outputs[0].text for o in outputs]
df['prediction'] = df['raw_output'].apply(extract_boxed)
df[['id', 'prediction']].to_csv('/kaggle/working/predictions.csv', index=False)
df[['id', 'raw_output', 'prediction']].to_csv('/kaggle/working/predictions_with_raw.csv', index=False)
print('saved predictions.csv ({} rows)'.format(len(df)))